# Day 08 — Pydantic + Database
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** `BaseModel` · `Field` constraints · `Optional` · `Literal` · `ValidationError` · `model_validate()` · `model_validate_json()` · `model_dump()` · connecting to PostgreSQL · `RealDictCursor` · parameterised queries · Pydantic + database together

> **Why Pydantic matters for LLM engineering:** The LLM returns a string. You parse it with `json.loads()`. But you still get a plain dict — no guarantees about which fields exist or what type they are. Pydantic validates the dict, converts types, and raises a clear error if anything is wrong — before the bad data reaches the rest of your code.


---


## 1 — Why Pydantic? The Problem with Plain Dicts

Without Pydantic you manually check every field after `json.loads()`:


In [1]:
import json

# LLM response — parsed with json.loads()
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'
data = json.loads(raw)

# You must manually check every field:
if 'category' not in data:
    raise ValueError('missing category')
if data['category'] not in ('TRACK_ORDER', 'CANCEL', 'REFUND'):
    raise ValueError(f'invalid category: {data["category"]}')
if 'confidence' not in data:
    raise ValueError('missing confidence')

# This gets tedious with 5+ fields, and easy to forget a check
print('Manual validation passed:', data)


Manual validation passed: {'category': 'TRACK_ORDER', 'confidence': 'high'}


**Pydantic does all of that in one line** — and gives you a typed object with IDE autocomplete.

| | Plain dict + manual checks | Pydantic `BaseModel` |
|-|---------------------------|---------------------|
| Type conversion | Manual | Automatic (`'4'` → `4`) |
| Missing fields | Manual check | Raises `ValidationError` |
| Invalid values | Manual check | Raises `ValidationError` |
| IDE autocomplete | None | Full autocomplete on fields |
| Error messages | You write them | Pydantic generates field-level messages |


---


## 2 — `BaseModel` — Your First Pydantic Model

Define a class that inherits from `BaseModel`. Each field is a class-level annotation.

```python
from pydantic import BaseModel

class MyModel(BaseModel):
    name: str
    age:  int
```

Pydantic automatically:
- **Validates** that `name` is a str and `age` is an int
- **Converts** types where possible (`'25'` → `25` for an int field)
- **Raises** a clear `ValidationError` with field-level messages on failure


In [8]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Literal

# The simplest Pydantic model — just field name and type
class CustomerRecord(BaseModel):
    customer_id : int
    first_name  : str
    last_name   : str
    email       : str
    city        : str
    state       : str


# Create an instance — Pydantic validates and converts types
customer = CustomerRecord(
    customer_id = 1001,
    first_name  = 'Prudhvi',
    last_name   = 'Akella',
    email       = 'prudhvi@example.com',
    city        = 'Rajahmundry',
    state       = 'AP',
)

print(customer)
print()
print('customer_id type:', type(customer.customer_id).__name__)  # int
print('first_name      :', customer.first_name)


customer_id=1001 first_name='Prudhvi' last_name='Akella' email='prudhvi@example.com' city='Rajahmundry' state='AP'

customer_id type: int
first_name      : Prudhvi


In [9]:
# Pydantic auto-converts types where possible
# CSV always gives strings — Pydantic converts '1001' → 1001 for int fields
from_csv = CustomerRecord(
    customer_id = '1001',   # str → Pydantic converts to int automatically
    first_name  = 'Prudhvi',
    last_name   = 'Akella',
    email       = 'prudhvi@example.com',
    city        = 'Rajahmundry',
    state       = 'AP',
)

print('customer_id:', from_csv.customer_id, '— type:', type(from_csv.customer_id).__name__)
# '1001' became int 1001 automatically


customer_id: 1001 — type: int


---


## 3 — `Field()` — Adding Constraints

`Field()` lets you add rules beyond just the type — min/max values, string lengths, descriptions.

| Constraint | Meaning | Example |
|-----------|---------|--------|
| `gt=0` | greater than 0 | `customer_id: int = Field(gt=0)` |
| `ge=0` | greater than or equal to 0 | `stock: int = Field(ge=0)` |
| `lt=n` | less than n | `rating: int = Field(lt=6)` |
| `le=5` | less than or equal to 5 | `rating: int = Field(le=5)` |
| `min_length=1` | string must have at least 1 char | `name: str = Field(min_length=1)` |
| `max_length=50` | string must have at most 50 chars | `name: str = Field(max_length=50)` |
| `default=value` | default if not provided | `country: str = Field(default='India')` |


In [10]:
class CustomerRecord(BaseModel):
    customer_id : int = Field(gt=0,            description='Must be a positive integer')
    first_name  : str = Field(min_length=1, max_length=50)
    last_name   : str = Field(min_length=1, max_length=50)
    email       : str = Field(min_length=5)
    city        : str
    state       : str = Field(min_length=2, max_length=2, description='Two-letter state code')
    country     : str = Field(default='India')   # default value

    @property
    def full_name(self):
        return f'{self.first_name} {self.last_name}'


# Valid — all constraints pass
c = CustomerRecord(
    customer_id=1001, first_name='Prudhvi', last_name='Akella',
    email='prudhvi@example.com', city='Rajahmundry', state='AP'
)
print(c.full_name, '|', c.country)   # country uses default 'India'


Prudhvi Akella | India


In [11]:
# Invalid — triggers ValidationError with clear field-level messages
try:
    bad = CustomerRecord(
        customer_id = -5,     # violates gt=0
        first_name  = '',     # violates min_length=1
        last_name   = 'Akella',
        email       = 'a@b.com',
        city        = 'Rajahmundry',
        state       = 'AndhraPradesh',  # violates max_length=2
    )
except ValidationError as e:
    print('ValidationError — field-level details:')
    for err in e.errors():
        print(f"  field: {err['loc']}  →  {err['msg']}")


ValidationError — field-level details:
  field: ('customer_id',)  →  Input should be greater than 0
  field: ('first_name',)  →  String should have at least 1 character
  field: ('state',)  →  String should have at most 2 characters


---


## 4 — `Optional` and `Literal`

### `Optional` — field that can be `None`

`Optional[str]` means the field can be a `str` OR `None`. Use it for fields that might not always exist.

### `Literal` — restrict to specific allowed values

`Literal['a', 'b', 'c']` means the value must be exactly one of those strings. If the LLM returns anything else → `ValidationError`.


In [12]:
class ProductRecord(BaseModel):
    product_id    : int   = Field(gt=0)
    product_name  : str   = Field(min_length=1)
    category      : str
    price         : float = Field(gt=0)
    stock_quantity: int   = Field(ge=0)           # 0 is valid (out of stock)
    description   : Optional[str] = None          # may or may not exist

    @property
    def is_in_stock(self):
        return self.stock_quantity > 0

    @property
    def price_str(self):
        return f'${self.price:.2f}'

    def __repr__(self):
        return f'ProductRecord(id={self.product_id}, name={self.product_name!r}, price={self.price_str})'


# description is None — that is fine
p1 = ProductRecord(product_id=101, product_name='Classic Monitor',
                   category='Electronics', price=205.21, stock_quantity=238)
print(p1)
print('in stock:', p1.is_in_stock)
print('description:', p1.description)   # None


product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238 description=None
in stock: True
description: None


In [13]:
# Literal — restrict to specific allowed values
class OrderRecord(BaseModel):
    order_id       : int
    customer_id    : int
    total_amount   : float = Field(ge=0)
    payment_method : str
    # Literal enforces that status must be one of these exact strings
    order_status   : Literal['Delivered', 'In Transit', 'Processing',
                              'Pending', 'Cancelled', 'Refunded']


# Valid status
o = OrderRecord(order_id=3042, customer_id=1001,
                total_amount=205.21, payment_method='Credit Card',
                order_status='In Transit')
print(o)

# Invalid status — LLM returned something unexpected
try:
    bad = OrderRecord(order_id=3043, customer_id=1001,
                      total_amount=50.0, payment_method='Cash',
                      order_status='Shipped')   # not in the Literal list
except ValidationError as e:
    print()
    print('Invalid status caught:')
    for err in e.errors():
        print(f"  {err['loc']}  →  {err['msg']}")


order_id=3042 customer_id=1001 total_amount=205.21 payment_method='Credit Card' order_status='In Transit'

Invalid status caught:
  ('order_status',)  →  Input should be 'Delivered', 'In Transit', 'Processing', 'Pending', 'Cancelled' or 'Refunded'


---


## 5 — `model_validate()` and `model_validate_json()`

| Method | Input | Use for |
|--------|-------|---------|
| `Model(field=value)` | keyword args | creating directly in code |
| `Model.model_validate(dict)` | Python dict | validating a CSV row or DB row |
| `Model.model_validate_json(string)` | JSON string | validating LLM response |


In [14]:
# model_validate() — validate a Python dict
# Use this when you have a dict (from csv.DictReader, database row, etc.)
row = {
    'product_id': '101',       # str — Pydantic converts to int
    'product_name': 'Classic Monitor',
    'category': 'Electronics',
    'price': '205.21',         # str — Pydantic converts to float
    'stock_quantity': '238',
}

product = ProductRecord.model_validate(row)
print(product)
print('price type  :', type(product.price).__name__)    # float (was str)
print('stock type  :', type(product.stock_quantity).__name__)  # int (was str)


product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238 description=None
price type  : float
stock type  : int


In [15]:
# model_validate_json() — parse AND validate a JSON string in one call
# This is what you use on LLM output
json_str = '{"product_id": 102, "product_name": "Yoga Mat", "category": "Sports", "price": 45.0, "stock_quantity": 150}'

product2 = ProductRecord.model_validate_json(json_str)
print(product2)


product_id=102 product_name='Yoga Mat' category='Sports' price=45.0 stock_quantity=150 description=None


---


## 6 — `model_dump()` — Convert Back to Dict

`model_dump()` converts a Pydantic object back into a plain Python dict.  
Use it when you need to pass data to `json.dumps()`, a database insert, or an API call.


In [16]:
product = ProductRecord(product_id=101, product_name='Classic Monitor',
                        category='Electronics', price=205.21, stock_quantity=238)

# model_dump() — back to a plain dict
d = product.model_dump()
print(type(d).__name__, d)

# model_dump_json() — back to a JSON string
s = product.model_dump_json()
print(type(s).__name__, s)

# Useful pattern: validate from DB row, modify, dump back for response
d['price'] = round(d['price'] * 0.9, 2)   # 10% discount
print('After discount:', d['price'])


dict {'product_id': 101, 'product_name': 'Classic Monitor', 'category': 'Electronics', 'price': 205.21, 'stock_quantity': 238, 'description': None}
str {"product_id":101,"product_name":"Classic Monitor","category":"Electronics","price":205.21,"stock_quantity":238,"description":null}
After discount: 184.69


---


## 7 — Using Pydantic to Validate LLM Output

This is the core LLM engineering use case. The system prompt tells the LLM to return JSON in a specific shape. Pydantic validates that shape.

**System prompt tells the LLM:**
```
## Output Format
Return JSON with exactly these fields:
{
  "category": one of TRACK_ORDER | CANCEL | REFUND | GENERAL,
  "confidence": one of high | medium | low,
  "reason": string explaining your classification
}
```


In [17]:
class TriageOutput(BaseModel):
    """
    Pydantic model for the LLM's classification output.
    Every field here matches a field the system prompt asks the LLM to return.
    """
    category   : Literal['TRACK_ORDER', 'CANCEL', 'REFUND', 'GENERAL']
    confidence : Literal['high', 'medium', 'low']
    reason     : str = Field(min_length=5)

    # model_config strips whitespace from all string fields automatically
    model_config = {'str_strip_whitespace': True}


def parse_llm_response(raw_response):
    """Strip markdown fences then validate through Pydantic."""
    cleaned = raw_response.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()

    # model_validate_json: parse JSON + validate in one call
    return TriageOutput.model_validate_json(cleaned)


# Test 1 — plain JSON
r1 = parse_llm_response('{"category": "TRACK_ORDER", "confidence": "high", "reason": "Customer asked about order location"}')
print('Test 1:', r1.category, r1.confidence)


Test 1: TRACK_ORDER high


In [18]:
# Test 2 — LLM wrapped in markdown fences (very common)
fenced = '```json\n{"category": "CANCEL", "confidence": "medium", "reason": "Customer wants to cancel"}\n```'
r2 = parse_llm_response(fenced)
print('Test 2:', r2.category, r2.confidence)


Test 2: CANCEL medium


In [19]:
# Test 3 — LLM returned invalid category
try:
    r3 = parse_llm_response('{"category": "UNKNOWN", "confidence": "high", "reason": "Not sure"}')
except ValidationError as e:
    print('Test 3 — caught invalid category:')
    for err in e.errors():
        print(f"  field {err['loc']} → {err['msg']}")


Test 3 — caught invalid category:
  field ('category',) → Input should be 'TRACK_ORDER', 'CANCEL', 'REFUND' or 'GENERAL'


In [20]:
# Test 4 — LLM missed a required field
try:
    r4 = parse_llm_response('{"category": "CANCEL", "confidence": "high"}')
    # 'reason' field missing
except ValidationError as e:
    print('Test 4 — missing field:')
    for err in e.errors():
        print(f"  field {err['loc']} → {err['msg']}")


Test 4 — missing field:
  field ('reason',) → Field required


---


## 8 — Database with psycopg2

`psycopg2` is the standard Python driver for PostgreSQL.

### Setup — create a `.env` file in your project root:

```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=shopsmart
DB_USER=postgres
DB_PASSWORD=yourpassword
```

### Key concepts

| Term | What it is |
|------|------------|
| **connection** | The open channel to the database |
| **cursor** | Used to send SQL and receive results |
| **`RealDictCursor`** | Returns rows as dicts `{column: value}` instead of plain tuples |
| **parameterised query** | Use `%s` placeholders — never string-format SQL (SQL injection) |
| **`commit()`** | Confirm INSERT/UPDATE/DELETE — changes aren't saved until you commit |


In [ ]:
import os
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv

load_dotenv()   # reads .env file into os.environ


def get_connection():
    """
    Open a connection to PostgreSQL using credentials from .env
    Always use inside a try/finally or with block to ensure it closes.
    """
    return psycopg2.connect(
        host    = os.environ.get('DB_HOST', 'localhost'),
        port    = int(os.environ.get('DB_PORT', '5432')),
        dbname  = os.environ.get('DB_NAME'),
        user    = os.environ.get('DB_USER'),
        password= os.environ.get('DB_PASSWORD'),
    )


# Test the connection
try:
    conn = get_connection()
    print('Connected to PostgreSQL')
    print('DB name:', conn.get_dsn_parameters().get('dbname'))
    conn.close()
except Exception as e:
    print(f'Connection failed: {e}')
    print('Check your .env file — DB_HOST, DB_NAME, DB_USER, DB_PASSWORD')


### Creating tables


In [ ]:
def create_tables():
    """Create the ShopSmart tables if they do not already exist."""
    conn = get_connection()

    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS customers (
                customer_id  INTEGER PRIMARY KEY,
                first_name   VARCHAR(50)  NOT NULL,
                last_name    VARCHAR(50)  NOT NULL,
                email        VARCHAR(100) UNIQUE NOT NULL,
                city         VARCHAR(50),
                state        VARCHAR(2),
                country      VARCHAR(50) DEFAULT 'India'
            )
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS products (
                product_id     INTEGER PRIMARY KEY,
                product_name   VARCHAR(200) NOT NULL,
                category       VARCHAR(100),
                price          NUMERIC(10,2) NOT NULL,
                stock_quantity INTEGER DEFAULT 0,
                description    TEXT
            )
        """)

    conn.commit()   # changes not saved until you commit
    conn.close()
    print('Tables created (or already exist)')


create_tables()


### INSERT — adding a row


In [ ]:
def insert_customer(customer_id, first_name, last_name, email, city, state, country='India'):
    """
    Insert one customer. ON CONFLICT DO NOTHING makes it safe to run twice.
    ALWAYS use %s placeholders — never f-strings in SQL (SQL injection risk).
    """
    conn = get_connection()

    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO customers (customer_id, first_name, last_name, email, city, state, country)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT DO NOTHING
        """, (customer_id, first_name, last_name, email, city, state, country))
        # ↑ tuple of values — psycopg2 replaces each %s safely

    conn.commit()
    conn.close()
    print(f'Inserted customer {customer_id}: {first_name} {last_name}')


insert_customer(1001, 'Prudhvi', 'Akella', 'prudhvi@example.com', 'Rajahmundry', 'AP')
insert_customer(1002, 'Ravi',    'Kumar',  'ravi@example.com',    'Hyderabad',   'TS')


### SELECT with `RealDictCursor`

`RealDictCursor` makes each row come back as a dict `{'column': 'value'}` instead of a plain tuple.  
This makes it very easy to pass directly into Pydantic's `model_validate()`.


In [ ]:
def get_all_customers():
    """Fetch all customers. RealDictCursor returns rows as dicts."""
    conn = get_connection()

    # cursor_factory=psycopg2.extras.RealDictCursor → rows as dicts
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute('SELECT * FROM customers ORDER BY customer_id')
        rows = cur.fetchall()   # list of dicts

    conn.close()
    return [dict(row) for row in rows]   # convert RealDictRow → plain dict


rows = get_all_customers()
for row in rows:
    print(row)


In [ ]:
def get_customer_by_id(customer_id):
    """Fetch one customer by ID. Returns None if not found."""
    conn = get_connection()

    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        # %s placeholder — psycopg2 handles quoting and escaping safely
        cur.execute('SELECT * FROM customers WHERE customer_id = %s', (customer_id,))
        row = cur.fetchone()   # one row or None

    conn.close()
    return dict(row) if row else None


row = get_customer_by_id(1001)
print('Raw row from DB:', row)


---


## 9 — Pydantic + Database Together

The pattern: fetch a raw dict from the database → validate it through a Pydantic model → work with a typed object.

This catches any bad data that made it into the database (wrong types, missing columns) before it reaches your application logic.


In [ ]:
def get_validated_customer(customer_id):
    """
    Fetch a customer and validate through Pydantic.
    Returns a CustomerRecord object or None.
    """
    row = get_customer_by_id(customer_id)

    if row is None:
        return None

    # model_validate() converts and validates the dict
    # If any column has wrong type/missing value → ValidationError
    return CustomerRecord.model_validate(row)


customer = get_validated_customer(1001)

if customer:
    print(f'Name       : {customer.full_name}')
    print(f'customer_id: {customer.customer_id}  (type: {type(customer.customer_id).__name__})')
    print(f'Email      : {customer.email}')
    print(f'Location   : {customer.city}, {customer.state}')


In [ ]:
# Full pipeline — DB row → Pydantic → f-string → LLM prompt
customer = get_validated_customer(1001)

if customer:
    user_message = (
        f'Customer  : {customer.full_name}\n'
        f'ID        : {customer.customer_id}\n'
        f'Location  : {customer.city}, {customer.state}\n'
        f'Query     : Where is my order #3042?'
    )
    print('User message ready for LLM:')
    print(user_message)


---


## Summary — Day 08

| Concept | Key rule |
|---------|----------|
| `BaseModel` | Inherit from it to create a validated data model |
| `Field(gt=0)` | Add constraints: gt, ge, lt, le, min_length, max_length |
| `Optional[str] = None` | Field can be str or None — not required |
| `Literal['a','b']` | Value must be exactly one of the listed strings |
| `ValidationError` | Raised with field-level messages when data is invalid |
| Type conversion | `'1001'` → `1001` for int fields — automatic |
| `Model.model_validate(dict)` | Validate a Python dict (from CSV, DB row) |
| `Model.model_validate_json(str)` | Parse JSON string and validate in one call |
| `model.model_dump()` | Convert Pydantic object back to plain dict |
| `model.model_dump_json()` | Convert to JSON string |
| `psycopg2.connect(...)` | Open a PostgreSQL connection (credentials from .env) |
| `conn.cursor()` | Create cursor to send SQL |
| `RealDictCursor` | Rows as dicts `{col: val}` instead of tuples |
| `cur.execute(sql, (val,))` | Always use %s — never f-strings in SQL |
| `cur.fetchone()` | One row or None |
| `cur.fetchall()` | List of all rows |
| `conn.commit()` | Confirm INSERT/UPDATE/DELETE |
| `conn.close()` | Always close the connection |
| Pydantic + DB pattern | `row = cur.fetchone()` → `Model.model_validate(dict(row))` |

**Next:** Day 09 — Async Python  
`async def` · `await` · `asyncio.gather()` · calling the OpenAI API asynchronously
